# Stable Diffusion 실행하기 (Hugging Face diffusers)

* Part A: StableDiffusionPipeline 한 줄로 이미지 생성
* Part B: 파이프라인을 네 조각(text encoder / U-Net / scheduler / VAE)으로 분해하여 denoising loop를 직접 작성


In [ ]:
import torch
from diffusers import StableDiffusionPipeline, DDIMScheduler

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
MODEL_ID = "runwayml/stable-diffusion-v1-5"   # SD 1.5 (512x512, 잠재 공간 64x64x4)

### Part A. 파이프라인으로 한 번에 생성

In [ ]:
def part_a(pipe):
    prompt = "a watercolor painting of a lighthouse at sunset, highly detailed"
    generator = torch.Generator(DEVICE).manual_seed(42)        # 재현성을 위한 seed
    image = pipe(
        prompt,
        num_inference_steps=30,    # denoising step 수 (DDIM이면 20~50이면 충분)
        guidance_scale=7.5,        # classifier-free guidance 강도 s
        generator=generator,
    ).images[0]
    image.save("part_a.png")
    print("saved part_a.png")

### Part B. 파이프라인 내부를 직접 구현

In [ ]:
@torch.no_grad()
def part_b(pipe, prompt, steps=30, guidance_scale=7.5, seed=42):
    tokenizer, text_encoder = pipe.tokenizer, pipe.text_encoder
    unet, vae, scheduler = pipe.unet, pipe.vae, pipe.scheduler

    # (1) 텍스트 인코딩: 조건 c 와 무조건(빈 문자열) 임베딩을 함께 준비 (CFG용)
    def encode(text):
        tokens = tokenizer(text, padding="max_length", 
                           max_length=tokenizer.model_max_length,
                           truncation=True, return_tensors="pt")\
                          .input_ids.to(DEVICE)
        return text_encoder(tokens)[0]                        # (1, 77, 768)

    cond_emb = encode(prompt)
    uncond_emb = encode("")
    text_emb = torch.cat([uncond_emb, cond_emb])              # (2, 77, 768) 배치로 묶기

    # (2) 초기 잠재 변수 z_T ~ N(0, I).  512x512 이미지 -> 64x64x4 잠재
    generator = torch.Generator(DEVICE).manual_seed(seed)
    latents = torch.randn((1, unet.config.in_channels, 64, 64),
                          generator=generator, device=DEVICE, dtype=DTYPE)
    scheduler.set_timesteps(steps)
    latents = latents * scheduler.init_noise_sigma           # scheduler별 스케일 보정

    # (3) Denoising loop: t = T ... 0
    for t in scheduler.timesteps:
        latent_in = torch.cat([latents] * 2)                 # uncond / cond 두 번 예측을 한 배치로
        latent_in = scheduler.scale_model_input(latent_in, t)

        eps = unet(latent_in, t, encoder_hidden_states=text_emb).sample
        eps_uncond, eps_cond = eps.chunk(2)

        # Classifier-free guidance: eps_hat = eps(∅) + s * (eps(c) - eps(∅))
        eps_hat = eps_uncond + guidance_scale * (eps_cond - eps_uncond)

        # scheduler.step 이 x_{t-1} 계산 (DDIM 수식)을 담당
        latents = scheduler.step(eps_hat, t, latents).prev_sample

    # (4) VAE 디코더로 잠재 -> 픽셀.  SD는 잠재를 0.18215 로 스케일링해 두었으므로 되돌림
    image = vae.decode(latents / vae.config.scaling_factor).sample
    image = (image / 2 + 0.5).clamp(0, 1)                    # [-1,1] -> [0,1]
    image = (image[0].permute(1, 2, 0).float().cpu().numpy() * 255).astype("uint8")

    from PIL import Image
    Image.fromarray(image).save("part_b.png")
    print("saved part_b.png")

### Part C. 실험 과제용 헬퍼: guidance scale 을 바꿔가며 grid 만들기

In [ ]:
def part_c(pipe, prompt, scales=(1.0, 3.0, 7.5, 15.0), seed=42):
    from diffusers.utils import make_image_grid
    images = []
    for s in scales:
        g = torch.Generator(DEVICE).manual_seed(seed)         # 같은 seed -> 같은 z_T
        images.append(pipe(prompt, num_inference_steps=30, guidance_scale=s, generator=g).images[0])
    make_image_grid(images, rows=1, cols=len(scales)).save("part_c_guidance.png")
    print("saved part_c_guidance.png  (scales:", scales, ")")

In [ ]:
pipe = StableDiffusionPipeline\
    .from_pretrained(MODEL_ID, torch_dtype=DTYPE).to(DEVICE)
pipe.scheduler = DDIMScheduler\
    .from_config(pipe.scheduler.config)   # DDIM 으로 교체

part_a(pipe)
part_b(pipe, "a photo of a corgi wearing sunglasses on the beach")
part_c(pipe, "an oil painting of a castle in the mountains")